In [ ]:
%pip install confluent-kafka  # required by job cluster until we deploy via DABs

In [ ]:
import json
import logging
import uuid
from pyspark.sql import Window
from pyspark.sql.functions import col, lit, row_number
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
logger = logging.getLogger("RepublishActiveEvents")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [ ]:
dbutils.widgets.dropdown(
    name="segment",
    defaultValue="APPEALS",
    choices=["APPEALS", "CDAM", "CASE_LINKING"],
    label="Segment"
)
segment = dbutils.widgets.get("segment")
print(f"Selected segment: {segment}")

In [ ]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key   = config.first()["lz_key"].strip().lower()
logger.info(f"env_name={env_name}, lz_key={lz_key}")

keyvault_name = f"ingest{lz_key}-meta002-{env_name}"
logger.info(f"keyvault_name={keyvault_name}")

In [ ]:
try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    tenant_id     = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    client_id     = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved all Service Principal secrets from Key Vault")
except Exception as e:
    logger.error(f"Failed to retrieve secrets: {e}", exc_info=True)
    raise

In [ ]:
curated_storage_account = f"ingest{lz_key}curated{env_name}"
try:
    configs = {
        f"fs.azure.account.auth.type.{curated_storage_account}.dfs.core.windows.net": "OAuth",
        f"fs.azure.account.oauth.provider.type.{curated_storage_account}.dfs.core.windows.net":
            "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{curated_storage_account}.dfs.core.windows.net": client_id,
        f"fs.azure.account.oauth2.client.secret.{curated_storage_account}.dfs.core.windows.net": client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{curated_storage_account}.dfs.core.windows.net":
            f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    }
    for key, val in configs.items():
        spark.conf.set(key, val)
    logger.info(f"Configured OAuth for {curated_storage_account}")
except Exception as e:
    logger.error(f"Failed to configure OAuth: {e}", exc_info=True)
    raise

In [ ]:
silver_base = f"abfss://silver@{curated_storage_account}.dfs.core.windows.net"

SEGMENT_CONFIG = {
    "APPEALS": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/publish_payload_audit",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit",
        "match_key":      "CaseNo",
        "topic":          f"evh-active-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
    "CDAM": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/CDAM/publish_audit_db_eh",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit",
        "match_key":      "CaseNo",
        "topic":          f"evh-active-cdam-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
    "CASE_LINKING": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/CASE_LINKING/publish_audit_db_eh",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit",
        "match_key":      "CCDCaseReferenceNumber",
        "topic":          f"evh-active-caselink-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
}

if segment not in SEGMENT_CONFIG:
    raise ValueError(f"Unknown segment '{segment}'. Valid options: {list(SEGMENT_CONFIG)}")

cfg = SEGMENT_CONFIG[segment]
print(f"pub_audit_path : {cfg['pub_audit_path']}")
print(f"ack_audit_path : {cfg['ack_audit_path']}")
print(f"topic          : {cfg['topic']}")

## Audit Comparison

In [ ]:
pub_df = spark.read.format("delta").load(cfg["pub_audit_path"])
published = (
    pub_df
    .filter(col("Status") == "SUCCESS")
    .select(col(cfg["match_key"]))
    .distinct()
)
published_count = published.count()
print(f"Successfully published: {published_count}")

In [ ]:
ack_df = spark.read.format("delta").load(cfg["ack_audit_path"])
acknowledged = (
    ack_df
    .filter(col("Status") == "SUCCESS")
    .select(col(cfg["match_key"]))
    .distinct()
)
acknowledged_count = acknowledged.count()
print(f"Successfully acknowledged: {acknowledged_count}")

In [ ]:
missing_keys = published.join(acknowledged, on=cfg["match_key"], how="left_anti").cache()
missing_count = missing_keys.count()
print(f"Published but not acknowledged: {missing_count}")

In [ ]:
missing_keys.display()

In [ ]:
if missing_count == 0:
    print("No missing events to republish. Exiting.")
    dbutils.notebook.exit("Nothing to republish — all published records have been acknowledged")

## Republish Missing Events

In [ ]:
eh_kv_secret = dbutils.secrets.get(scope=keyvault_name, key="RootManageSharedAccessKey")
eventhubs_hostname = f"ingest{lz_key}-integration-eventHubNamespace001-{env_name}.servicebus.windows.net:9093"

kafka_conf = {
    'bootstrap.servers': eventhubs_hostname,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',
    'sasl.password': eh_kv_secret,
    'retries': 5,
    'enable.idempotence': True,
    'queue.buffering.max.messages': 500000,
    'queue.buffering.max.kbytes': 2097152,
}
broadcast_kafka = sc.broadcast(kafka_conf)
broadcast_meta  = sc.broadcast({"topic": cfg["topic"], "segment": segment})
print(f"Event Hub topic: {cfg['topic']}")

In [ ]:
try:
    context_str = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
    context     = json.loads(context_str)
    run_id      = context.get("tags", {}).get("jobRunId") or str(uuid.uuid4())
except Exception:
    run_id = str(uuid.uuid4())

if segment == "APPEALS":
    w = Window.partitionBy("CaseNo").orderBy(col("PublishingDateTime").desc())
    missing_data = (
        pub_df
        .filter(col("Status") == "SUCCESS")
        .join(missing_keys, on="CaseNo", how="inner")
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
        .select("CaseNo", "AriaCaseNo", "State", "PublishContent")
        .withColumn("RunID", lit(run_id))
    )

elif segment == "CDAM":
    w = Window.partitionBy("CaseNo").orderBy(col("PublishingDateTime").desc())
    missing_data = (
        pub_df
        .filter(col("Status") == "SUCCESS")
        .join(missing_keys, on="CaseNo", how="inner")
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
        .select("CaseNo", "FileName", "FileURL")
        .withColumn("RunID", lit(run_id))
    )

elif segment == "CASE_LINKING":
    payload_df = spark.read.table("hive_metastore.ariadm_active_appeals.case_link_payloads")
    missing_data = (
        missing_keys
        .join(
            payload_df.withColumnRenamed("CCDCaseReferenceNumberFrom", "CCDCaseReferenceNumber"),
            on="CCDCaseReferenceNumber",
            how="inner"
        )
        .select("CCDCaseReferenceNumber", col("caseLinks").alias("CaseLinkPayload"))
        .withColumn("RunID", lit(run_id))
    )

missing_data.display()

In [ ]:
result_schema_map = {
    "APPEALS": StructType([
        StructField("RunID",            StringType(), True),
        StructField("AriaCaseNo",       StringType(), True),
        StructField("CaseNo",           StringType(), True),
        StructField("Filename",         StringType(), True),
        StructField("State",            StringType(), True),
        StructField("PublishingDateTime", StringType(), True),
        StructField("Status",           StringType(), True),
        StructField("PublishContent",   StringType(), True),
        StructField("Error",            StringType(), True),
    ]),
    "CDAM": StructType([
        StructField("RunID",            StringType(), True),
        StructField("CaseNo",           StringType(), True),
        StructField("FileName",         StringType(), True),
        StructField("FileURL",          StringType(), True),
        StructField("PublishingDateTime", StringType(), True),
        StructField("Status",           StringType(), True),
        StructField("Error",            StringType(), True),
    ]),
    "CASE_LINKING": StructType([
        StructField("RunID",                  StringType(), True),
        StructField("CCDCaseReferenceNumber", StringType(), True),
        StructField("CaseLinkCount",          StringType(), True),
        StructField("PublishingDateTime",     StringType(), True),
        StructField("Status",                 StringType(), True),
        StructField("Error",                  StringType(), True),
    ]),
}
result_schema = result_schema_map[segment]


def process_partition(partition):
    import json
    from confluent_kafka import Producer
    from datetime import datetime, timezone

    kafka_conf = dict(broadcast_kafka.value)
    topic      = broadcast_meta.value["topic"]
    seg        = broadcast_meta.value["segment"]

    producer = Producer(**kafka_conf)
    results  = []

    def make_callback(rd, ts):
        def callback(err, msg):
            status = "ERROR" if err else "SUCCESS"
            error  = str(err) if err else ""
            if seg == "APPEALS":
                results.append((
                    rd["RunID"], rd.get("AriaCaseNo", ""), rd["CaseNo"], rd["CaseNo"],
                    rd["State"], ts, status, rd.get("PublishContent", "") if not err else "", error
                ))
            elif seg == "CDAM":
                results.append((
                    rd["RunID"], rd["CaseNo"], rd.get("FileName", ""), rd.get("FileURL", ""),
                    ts, status, error
                ))
            elif seg == "CASE_LINKING":
                payload = rd.get("CaseLinkPayload") or []
                results.append((
                    rd["RunID"], rd["CCDCaseReferenceNumber"], str(len(payload)), ts, status, error
                ))
        return callback

    for row in partition:
        ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
        rd = row.asDict(recursive=True)
        producer.poll(0)

        if seg == "APPEALS":
            key   = (rd.get("CaseNo") or "").encode("utf-8")
            value = json.dumps({
                "RunID":   rd.get("RunID"),
                "CaseNo":  rd.get("CaseNo"),
                "State":   rd.get("State"),
                "Content": rd.get("PublishContent"),
            }).encode("utf-8")
        elif seg == "CDAM":
            key   = (rd.get("CaseNo") or "").encode("utf-8")
            value = json.dumps({
                "RunID":    rd.get("RunID"),
                "CaseNo":   rd.get("CaseNo"),
                "FileName": rd.get("FileName"),
                "FileURL":  rd.get("FileURL"),
            }).encode("utf-8")
        elif seg == "CASE_LINKING":
            key   = (rd.get("CCDCaseReferenceNumber") or "").encode("utf-8")
            value = json.dumps({
                "RunID":                  rd.get("RunID"),
                "CCDCaseReferenceNumber": rd.get("CCDCaseReferenceNumber"),
                "CaseLinkPayload": {
                    "caseLinks":               rd.get("CaseLinkPayload"),
                    "isNotificationTurnedOff": "Yes",
                },
            }).encode("utf-8")
        else:
            continue

        cb = make_callback(rd, ts)
        try:
            producer.produce(topic=topic, key=key, value=value, callback=cb)
        except BufferError:
            producer.poll(10)
            try:
                producer.produce(topic=topic, key=key, value=value, callback=cb)
            except BufferError:
                producer.poll(30)
                try:
                    producer.produce(topic=topic, key=key, value=value, callback=cb)
                except BufferError as e:
                    ts2 = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
                    if seg == "APPEALS":
                        results.append((rd.get("RunID"), rd.get("AriaCaseNo", ""), rd.get("CaseNo"), rd.get("CaseNo"), rd.get("State"), ts2, "ERROR", "", str(e)))
                    elif seg == "CDAM":
                        results.append((rd.get("RunID"), rd.get("CaseNo"), rd.get("FileName", ""), rd.get("FileURL", ""), ts2, "ERROR", str(e)))
                    elif seg == "CASE_LINKING":
                        payload = rd.get("CaseLinkPayload") or []
                        results.append((rd.get("RunID"), rd.get("CCDCaseReferenceNumber"), str(len(payload)), ts2, "ERROR", str(e)))

    try:
        producer.flush()
    except Exception as e:
        logger.error(f"Flush error: {e}")

    return results


optimized_df = missing_data.repartition(8)
result_rdd   = optimized_df.rdd.mapPartitions(process_partition)
result_df    = spark.createDataFrame(result_rdd, result_schema).cache()

result_df.display()

In [ ]:
failed_df = result_df.filter(col("Status") == "ERROR")
failed_df.display()
print(f"Failed: {failed_df.count()}")

In [ ]:
result_df.write.format("delta").mode("append").option("mergeSchema", "true").save(cfg["pub_audit_path"])
logger.info(f"Republish audit appended to {cfg['pub_audit_path']}")

In [ ]:
successful = result_df.filter(col("Status") == "SUCCESS").count()
failed     = result_df.filter(col("Status") == "ERROR").count()

print(f"Segment          : {segment}")
print(f"Missing events   : {missing_count}")
print(f"Republish success: {successful}")
print(f"Republish failed : {failed}")

dbutils.notebook.exit({"segment": segment, "missing": missing_count, "republish_success": successful, "republish_failed": failed})